In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Loading packages, tokenizers and defining the evaluation set and gdrive links
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import os


MODEL_PATH = "/content/drive/MyDrive/cultural-recipes-data/final"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

EVAL_SETS = {
    "human_cn2en":  "/content/drive/MyDrive/cultural-recipes-data/human_cn2en/sampled_human_mt5_base_cn2en.jsonl",
    "human_en2cn":  "/content/drive/MyDrive/cultural-recipes-data/human_en2cn/sampled_human_mt5_base_en2cn.jsonl",
    "silver_cn2en": "/content/drive/MyDrive/cultural-recipes-data/silver_cn2en/sampled_silver_mt5_base_cn2en.jsonl",
    "silver_en2cn": "/content/drive/MyDrive/cultural-recipes-data/silver_en2cn/sampled_silver_mt5_base_en2cn.jsonl",
}

final_path = "/content/drive/MyDrive/cultural-recipes-data/final"


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
!pip install -q rouge-score sacrebleu

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch, os, json

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.0 MB/s eta 0:00:00


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [7]:
#Function definition

from rouge_score import rouge_scorer
import sacrebleu

def generate_prediction(source_text, max_length=256):
    inputs = tokenizer(
        source_text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=5,
            early_stopping=True,
            no_repeat_ngram_size=4
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def compute_metrics(predictions, references, is_chinese_target=False):
    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references],
        tokenize="char" if is_chinese_target else "13a"
    )
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=False
    )
    r1, r2, rL = [], [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rL.append(scores["rougeL"].fmeasure)
    return {
        "bleu":   round(bleu.score, 2),
        "rouge1": round(sum(r1) / len(r1) * 100, 2),
        "rouge2": round(sum(r2) / len(r2) * 100, 2),
        "rougeL": round(sum(rL) / len(rL) * 100, 2),
    }

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


In [10]:
#Generation Evaluation Metrics

BASE = "/content/drive/MyDrive/cultural-recipes-data"

EVAL_SETS = {
    "human_cn2en":  (f"{BASE}/human_cn2en/sampled_human_mt5_base_cn2en.jsonl",  False),
    "human_en2cn":  (f"{BASE}/human_en2cn/sampled_human_mt5_base_en2cn.jsonl",  True),
    "silver_cn2en": (f"{BASE}/silver_cn2en/sampled_silver_mt5_base_cn2en.jsonl", False),
    "silver_en2cn": (f"{BASE}/silver_en2cn/sampled_silver_mt5_base_en2cn.jsonl", True),
}

all_results = {}
model.eval()

for name, (path, is_chinese_target) in EVAL_SETS.items():
    print(f"\n{name}")
    records = load_jsonl(path)
    print(f"Records: {len(records)}")

    our_preds      = []
    baseline_preds = []
    references     = []

    for i, record in enumerate(records):
        references.append(record["references"][0])
        baseline_preds.append(record["prediction"])
        our_preds.append(generate_prediction(record["source"]))


    our_scores      = compute_metrics(our_preds, references, is_chinese_target)
    baseline_scores = compute_metrics(baseline_preds, references, is_chinese_target)

    all_results[name] = {
        "our_scores": our_scores,
        "baseline_scores": baseline_scores,
        "our_predictions": our_preds,
        "references": references,
        "records": records,
    }

    print(f"\n  {'Metric':<10} {'mT5-small':>15} {'Paper mT5-base':>15} {'Diff':>10}")
    for metric in ["bleu", "rouge1", "rouge2", "rougeL"]:
        ours  = our_scores[metric]
        paper = baseline_scores[metric]
        diff  = ours - paper
        sign  = "+" if diff > 0 else ""
        print(f"  {metric:<10} {ours:>15.2f} {paper:>15.2f} {sign}{diff:>9.2f}")


human_cn2en
Records: 25

  Metric           mT5-small  Paper mT5-base       Diff
  bleu                  4.19            1.60 +     2.59
  rouge1               30.39           25.26 +     5.13
  rouge2                6.41            4.29 +     2.12
  rougeL               22.39           17.84 +     4.55

human_en2cn
Records: 41

  Metric           mT5-small  Paper mT5-base       Diff
  bleu                 11.68           15.20     -3.52
  rouge1               35.66           32.97 +     2.69
  rouge2               13.43           11.93 +     1.50
  rougeL               33.49           31.69 +     1.80

silver_cn2en
Records: 82

  Metric           mT5-small  Paper mT5-base       Diff
  bleu                  3.94            3.74 +     0.20
  rouge1               29.31           28.93 +     0.38
  rouge2                7.69            6.92 +     0.77
  rougeL               20.91           19.51 +     1.40

silver_en2cn
Records: 52

  Metric           mT5-small  Paper mT5-base       Diff

In [11]:
print("\n=== FULL SUMMARY ===\n")
print(f"{'Set':<15} {'Metric':<10} {'mT5-small':>15} {'Paper mT5-base':>15} {'Diff':>10}")

for name, result in all_results.items():
    for metric in ["bleu", "rouge1", "rouge2", "rougeL"]:
        ours  = result["our_scores"][metric]
        paper = result["baseline_scores"][metric]
        diff  = ours - paper
        sign  = "+" if diff > 0 else ""
        print(f"{name:<15} {metric:<10} {ours:>15.2f} {paper:>15.2f} {sign}{diff:>9.2f}")
    print()


=== FULL SUMMARY ===

Set             Metric           mT5-small  Paper mT5-base       Diff
human_cn2en     bleu                  4.19            1.60 +     2.59
human_cn2en     rouge1               30.39           25.26 +     5.13
human_cn2en     rouge2                6.41            4.29 +     2.12
human_cn2en     rougeL               22.39           17.84 +     4.55

human_en2cn     bleu                 11.68           15.20     -3.52
human_en2cn     rouge1               35.66           32.97 +     2.69
human_en2cn     rouge2               13.43           11.93 +     1.50
human_en2cn     rougeL               33.49           31.69 +     1.80

silver_cn2en    bleu                  3.94            3.74 +     0.20
silver_cn2en    rouge1               29.31           28.93 +     0.38
silver_cn2en    rouge2                7.69            6.92 +     0.77
silver_cn2en    rougeL               20.91           19.51 +     1.40

silver_en2cn    bleu                 11.19           16.71     -5

In [12]:
print("Example Translations\n")

for name, result in all_results.items():
    print(f"{name}")
    for i in range(min(2, len(result["records"]))):
        record = result["records"][i]
        print(f"\nExample {i+1}:")
        print(f"  SOURCE:     {record['source'][:120]}...")
        print(f"  REFERENCE:  {result['references'][i][:120]}...")
        print(f"  PAPER:      {record['prediction'][:120]}...")
        print(f"  OURS:       {result['our_predictions'][i][:120]}...")
    print()

Example Translations

human_cn2en

Example 1:
  SOURCE:     Title: 星洲炒米粉, Ingredients: 3卷星洲炒米专用米粉 1520只鲜虾 大半个洋葱 一小根胡萝卜 35个青椒 1大根火腿肠 适量蒜末大葱段干辣椒 适量咖喱粉 适量盐 适量食用油 适量蚝油, Steps: 以上是所需要的食...
  REFERENCE:  Title: Chinese stir fry with rice noodles, Ingredients: 1. 100g rice vermicelli 2. 300g peeled and frozen shrimp, thawed...
  PAPER:      title: Hawaiian Fried Rice; ingredient: 1 c. uncooked rice 1 c. chopped celery 1 c. sliced carrots 1 c. diced potatoes 1...
  OURS:       Title: Thai Fried Rice, Ingredients: 2 tablespoons olive oil, 1 medium onion, chopped, 3 garlic cloves, minced, 1 teaspo...

Example 2:
  SOURCE:     Title: 扬州炒饭, Ingredients: 半根胡萝卜 两个鸡蛋 半根黄瓜 小半个紫皮圆葱 半根秋林红肠, Steps: 紫皮圆葱、黄瓜、红肠切小丁，胡萝卜切沫，备用 鸡蛋打散，用筷子高速搅打几下至出沫，下油锅炒散，盛出来，备用。炒...
  REFERENCE:  Title: egg fried rice, Ingredients: half a carrot 2 eggs half a cucumber half small red onion half a sausage cooked rice...
  PAPER:      title: Shanghai Fried Rice; ingredient: 1 cup long-grain white rice 2 tablespoons peanut oil 2 table